# NB7: Debugging Broken Agents

**2-minute intro script:** Learners need to see failures before production. This sandbox intentionally triggers schema drift, tool overreach, unbounded repair risk, and memory leakage. The point is diagnostic muscle: when an agent system behaves strangely, check the handoff schema, the tool policy, the retry budget, and the memory filter first.

In [ ]:
from pydantic import BaseModel, ConfigDict, ValidationError

class StrictPatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    path: str
    content: str

try:
    StrictPatch(path="app.py", content="print(1)", secret="leak")
except ValidationError as exc:
    print("Schema drift blocked:", exc.errors()[0]["type"])

In [ ]:
allowed = {"coder": {"read_repo"}, "qa": {"run_tests"}}

def authorize(role, tool):
    if tool not in allowed.get(role, set()):
        raise PermissionError(f"{role} cannot use {tool}")

try:
    authorize("coder", "run_tests")
except PermissionError as exc:
    print("Tool overreach blocked:", exc)

In [ ]:
def unsafe_memory_search(records, query):
    return [r for r in records if query.lower() in r.lower()]

def safe_memory_search(records, query, requester):
    return [
        r["text"]
        for r in records
        if requester in r["visible_to"] and query.lower() in r["text"].lower()
    ]

records = [
    {"text": "Public design note", "visible_to": {"coder", "qa"}},
    {"text": "Production DB password: super_secret_123", "visible_to": {"security"}},
]

print("Unsafe search leaks:", unsafe_memory_search([r["text"] for r in records], "password"))
print("Safe search returns:", safe_memory_search(records, "password", "coder"))

In [ ]:
max_repairs = 2
for attempt in range(max_repairs + 1):
    print("repair attempt", attempt)
else:
    print("Escalate instead of looping forever")

## 🧪 Exercises: The Broken-Agent Sandbox

**The Story:** Your capstone team is almost ready to ship. Then the Coder adds a secret field, the QA agent tries to use a tool it does not own, the memory search leaks a password, and the repair loop keeps retrying. This is not a disaster if the system is observable. It is only a disaster if no one knows where the failure entered the pipeline.

**Your Mission:**
1. **Schema Drift Drill:** Add a failing example for invalid enum values and prove Pydantic blocks it before downstream agents consume it.
2. **Unauthorized Tool Drill:** Add an audit log for denied tool calls. The log must include `role`, `tool`, `allowed=False`, and `reason`.
3. **Memory Leakage Drill:** Create a memory leakage test that fails with `unsafe_memory_search` and passes with `safe_memory_search`.
4. **Repair Budget Drill:** Add an `EscalationTicket` schema for exhausted repair loops instead of returning loose strings.
5. **Capstone Readiness Drill:** Write a debugging checklist for your capstone team: schema, tool, memory, repair, route, API boundary.

**The Takeaway:** Debugging multi-agent teams is management work. When something fails, inspect the contract, the boundary, the memory filter, and the retry budget before blaming the model.